# Dataset Description
In this notebook, we are using a dataset containing daily financial market data for multiple U.S. stocks and financial assets, including AMD, GLD, GS, INTC, JPM, META, MSFT, MU, NVDA, RXRX, and TSLA. The stocks time range are from 2018 to 2025, except a new stock RXRX from 2021 to 2025.

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Load data
stock_symbols = ['AMD', 'GLD', 'GS', 'INTC', 'JPM', 'META', 'MSFT', 'MU', 'NVDA', 'RXRX', 'TSLA']
path = 'Pure_data/' 
stock_data = {} 

for symbol in stock_symbols: 
    try: 
        file_path = f'{path}{symbol}.csv' 
        df = pd.read_csv(file_path) 
        stock_data[symbol] = df 
        print(f"Successfully loaded {symbol}.csv") 
    except FileNotFoundError: 
        print(f"Error: {symbol}.csv not found at {file_path}") 
    except Exception as e: 
        print(f"Error loading {symbol}.csv: {e}") 
        
print("\nLoaded stock symbols:")

print(stock_data.keys())

Successfully loaded AMD.csv
Successfully loaded GLD.csv
Successfully loaded GS.csv
Successfully loaded INTC.csv
Successfully loaded JPM.csv
Successfully loaded META.csv
Successfully loaded MSFT.csv
Successfully loaded MU.csv
Successfully loaded NVDA.csv
Successfully loaded RXRX.csv
Successfully loaded TSLA.csv

Loaded stock symbols:
dict_keys(['AMD', 'GLD', 'GS', 'INTC', 'JPM', 'META', 'MSFT', 'MU', 'NVDA', 'RXRX', 'TSLA'])


In [3]:
stock_data['AMD'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11572 entries, 0 to 11571
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    11571 non-null  object
 1   Close   11572 non-null  object
 2   High    11572 non-null  object
 3   Low     11572 non-null  object
 4   Open    11572 non-null  object
 5   Volume  11572 non-null  object
 6   symbol  11571 non-null  object
dtypes: object(7)
memory usage: 633.0+ KB


# Data Preprocessing
we are going to use columns close, high, low, open, and volumn to calculate features that are important in our stock prediction.

## Exponential Moving Average (EMA)
The exponential moving average is a type of moving average that gives more weight to recent data points and less data to older data points.

To be more specific, $$\text{EMA}_t = \alpha * P_t + (1-\alpha) * \text{EMA}_{t-1}$$
such that 
- $P_t=$ current price
- $\text{EMA}_{t-1}=$ previous EMA value
- $\alpha=$ Smoothing factor

shorter EMA $\to$ reacts faster to price changes

In [4]:
def compute_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

## Simple Moving Average (SMA)
The simple moving average calculates the average of the most recent observations.
$$\text{SMA}_t = \frac{1}{n}\sum_{i=1}^n x_i$$

- price above SMA $\to$ Uptrend
- price below SMA $\to$ Downtrend

In [5]:
def compute_sma(series, window):
    return series.rolling(window=window).mean()

## Relative Strength Index (RSI)
The RSI measures the ratio of average gains to average losses over a given period.
$$\begin{align}
\text{RS}=\frac{\text{Average Gain}}{\text{Average Loss}}\\
\text{RSI}=100-\frac{100}{1+\text{RS}}
\end{align}$$
- If prices go up a lot $\to$ Average gain large $\to$ RSI high
- If prices go down a lot $\to$ Average loss large $\to$ RSI low
- If balanced $\to$ RSI near 50

In [6]:
def compute_rsi(series, window):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

## Moving Average Convergence Divergence (MACD)
MACD is a trend-following momentum indictor used in technical analysis to understand trend direction and make sure buy or sell signal. It is the difference between two EMAs.

$$\text{MACD}=\text{EMA}_{12}-\text{EMA}_{26}$$

- MACD > 0 $\to$ Upward Momentum
- MACD < 0 $\to$ Downward Momentum

MACD usually have three components:

- MACD Line: $\text{MACD}=\text{EMA}_{12}-\text{EMA}_{26}$
- Signal Line: $\text{Signal}=\text{EMA}_9(\text{MACD})$
- MACD Histogram $\text{Histogram}=\text{MACD}-\text{Signal}$

In [7]:
def compute_macd(price, fast=12, slow=26):
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    return macd

def compute_signal(macd, signal_period=9):
    signal = macd.ewm(span=signal_period, adjust=False).mean()
    return signal

def compute_histogram(macd, signal):
    histogram = macd - signal
    return histogram

## Average True Range (ATR)
ATR is a technical indicator that measures market volatility — how much the price moves on average. To be more specific, ATR does not tell the direction of the moving, but just how much the prices moving.

$$\text{TR}=\max \left\{
\begin{aligned}
&\text{High} - \text{Low} \\
&\left| \text{High} - \text{Previous Close} \right| \\
&\left| \text{Low} - \text{Previous Close} \right|
\end{aligned}
\right.$$

$$\text{ATR}_n = \text{Moving average of TR over n periods}$$

In [8]:
def compute_atr(high, low, close, window):
    prev_close = close.shift(1)

    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/window, adjust=False).mean()
    return atr

## Volumn Simple Moving Average (Volumn_SMA)
Volume_SMA (Volume Simple Moving Average) is the simple moving average of trading volume over a specified number of periods.

$$\text{Volumn_SMA}_t=\frac{1}{N}\sum_{i=0}^{N-1}\text{Volumn}_{t-i}$$

In [9]:
def volume_sma(volume, window):
    return volume.rolling(window=window).mean()

## On Balance Volumn (OBV)
On-Balance Volume (OBV) is a technical indicator that uses volume flow to predict price movement.
$$OBV_t =
\begin{cases}
OBV_{t-1} + Volume_t & \text{if } Close_t > Close_{t-1} \\
OBV_{t-1} - Volume_t & \text{if } Close_t < Close_{t-1} \\
OBV_{t-1} & \text{otherwise}
\end{cases}$$

In [10]:
def compute_obv(close, volume):
    obv = [0]
    
    for i in range(1, len(close)):
        if close[i] > close[i-1]:
            obv.append(obv[-1] + volume[i])
        elif close[i] < close[i-1]:
            obv.append(obv[-1] - volume[i])
        else:
            obv.append(obv[-1])
    
    return pd.Series(obv, index=close.index)

## On Balance Volumn Z Score (OBV_Z_Score)
OBV_Z_Score measures how far the current OBV is from its average, in units of standard deviation.
$$\text{OBV Z-score}
=
\frac{
OBV - \text{Mean}(OBV_{\text{window}})
}{
\text{Std}(OBV_{\text{window}})
}$$
- OBV = On Balance Volumn

In [11]:
def compute_obv_zscore(obv, window=20):
    mean = obv.rolling(window).mean()
    std = obv.rolling(window).std()
    
    zscore = (obv - mean) / std
    return zscore

## K & D (Stochastic Oscillator)
The Stochastic Oscillator is a momentum indicator used in technical analysis to determine overbought and oversold conditions. 

It consists of two lines:
- %K → Fast line (main indicator)
- %D → Smoothed signal line

$$\%K=\frac{C-L_n}{H_n-L_n}\times 100\%$$
- $C$ is the current closing price
- $L_n$ is the lowest price over last $n$ periods
- $H_n$ is the highest price over last $n$ periods

$$\%D = \text{SMA}_3(\%K)$$

Usually, if $\%K$ crosses $\%D$ above, it is a singal of buying; If $\%K$ crosses $\%D$ below, it is a signal of selling.

In [12]:
def stochastic_k(df, k_period=14):
    low_min = df['Low'].rolling(window=k_period).min()
    high_max = df['High'].rolling(window=k_period).max()
    k = 100 * (df['Close'] - low_min) / (high_max - low_min)
    return k

def stochastic_d(k_values, d_period=3):
    d = k_values.rolling(window=d_period).mean()
    return d

## Money Flow Index (MFI)
The Money Flow Index (MFI) is a momentum indicator that measures buying and selling pressure using price AND volume. 

To calculate money flow index, we first need to calculate money flow and money flow ratio.

$$\text{Money Flow} = \frac{\text{High} + \text{Low} + \text{Close}}{3} * \text{Volumn}$$

$$\text{Money Flow Ratio}=\frac{\text{Positive Money Flow}}{\text{Negative Money Flow}}$$

Hence, money flow index is 

$$\text{MFI}=100-\frac{100}{1+\text{Money Flow Ratio}}$$

- if $\text{MFI}>80$ $\to$ overbought (price may fall)
- if $\text{MFI}<20$ $\to$ oversold (price may rise)

In [13]:
def compute_mfi(high, low, close, volume, window):
    typical_price = (high + low + close) / 3
    money_flow = typical_price * volume

    tp_diff = typical_price.diff()

    positive_flow = money_flow.where(tp_diff > 0, 0.0)
    negative_flow = money_flow.where(tp_diff < 0, 0.0)

    positive_mf = positive_flow.rolling(window=window).sum()
    negative_mf = negative_flow.rolling(window=window).sum()

    money_ratio = positive_mf / negative_mf
    mfi = 100 - (100 / (1 + money_ratio))
    return mfi

## Bollinger Bands
Bollinger Bands are a technical indicator used in trading to measure volatility and identify overbought/oversold conditions. It consists of three lines:
- Middle Band $\to$ Moving average (usually 20-day SMA)
- Upper Band $\to$ Middle Band + 2 * Standard Deviation
- Lower Band $\to$ Middle Band - 2 * Standard Deviation

$$\text{MB}=\text{SMA}_{20}$$
$$\text{UB}=\text{SMA}_{20}+2\sigma$$
$$\text{LB}=\text{SMA}_{20}-2\sigma$$
with $\sigma$ be standard deviation of price

In [14]:
def bollinger_bands(price, window=20):
    sma = price.rolling(window).mean()
    std = price.rolling(window).std()
    
    upper_band = sma + 2 * std
    lower_band = sma - 2 * std
    
    return sma, upper_band, lower_band

## Returns
Returns measure how much you gain or lose from an investment over a period of time.

$$\text{Return}=\frac{P_t-P_{t-k}}{P_{t-k}}$$
where $P_t$ is the current price and $P_{t-k}$ is the previous price

In [15]:
def compute_return(price, period=1):
    return price.pct_change(periods=period)

# Computing these features

In [16]:
feature_data = {}

for symbol, df in stock_data.items():

    df = df.copy()

    # clean data
    df = df[df["Date"].notna()].copy()
    df["Date"] = pd.to_datetime(df["Date"])

    numeric_cols = ["Close", "High", "Low", "Open", "Volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("Date").reset_index(drop=True)

    # =========================
    # Compute Features
    # =========================

    # EMA
    df["EMA_5"] = compute_ema(df["Close"], 5)
    df["EMA_10"] = compute_ema(df["Close"], 10)
    df["EMA_20"] = compute_ema(df["Close"], 20)
    df["EMA_50"] = compute_ema(df["Close"], 50)

    # SMA
    df["SMA_50"] = compute_sma(df["Close"], 50)
    df["SMA_100"] = compute_sma(df["Close"], 100)
    df["SMA_200"] = compute_sma(df["Close"], 200)

    # RSI
    df["RSI_5"] = compute_rsi(df["Close"], 5)
    df["RSI_14"] = compute_rsi(df["Close"], 14)
    df["RSI_21"] = compute_rsi(df["Close"], 21)

    # MACD
    # MACD
    df["MACD"] = compute_macd(df["Close"])
    df["Signal_Line"] = compute_signal(df["MACD"])
    df["MACD_Histogram"] = compute_histogram(df["MACD"], df["Signal_Line"])

    # ATR
    df["ATR_5"] = compute_atr(df["High"], df["Low"], df["Close"], 5)
    df["ATR_14"] = compute_atr(df["High"], df["Low"], df["Close"], 14)
    df["ATR_21"] = compute_atr(df["High"], df["Low"], df["Close"], 21)

    # Volume SMA
    df["Volume_SMA_5"] = compute_sma(df["Volume"], 5)
    df["Volume_SMA_20"] = compute_sma(df["Volume"], 20)
    df["Volume_SMA_60"] = compute_sma(df["Volume"], 60)

    # OBV
    df["OBV"] = compute_obv(df["Close"], df["Volume"])
    df["OBV_Z_Score_20"] = compute_obv_zscore(df["OBV"], 20)
    df["OBV_Z_Score_50"] = compute_obv_zscore(df["OBV"], 50)

    # K 5
    df["K_5"] = stochastic_k(df, k_period=5)
    df["D_5"] = stochastic_d(df["K_5"], d_period=3)

    # K 14
    df["K_14"] = stochastic_k(df, k_period=14)
    df["D_14"] = stochastic_d(df["K_14"], d_period=3)

    # MFI
    df["MFI_5"] = compute_mfi(df["High"], df["Low"], df["Close"], df["Volume"], 5)
    df["MFI_14"] = compute_mfi(df["High"], df["Low"], df["Close"], df["Volume"], 14)
    df["MFI_21"] = compute_mfi(df["High"], df["Low"], df["Close"], df["Volume"], 21)

    # Bollinger Bands
    df["BB_Middle_20"], df["BB_Upper_20"], df["BB_Lower_20"] = bollinger_bands(df["Close"], 20)

    # Returns
    df["1d_return"] = compute_return(df["Close"], 1)
    df["5d_return"] = compute_return(df["Close"], 5)
    df["10d_return"] = compute_return(df["Close"], 10)
    df["20d_return"] = compute_return(df["Close"], 20)
    df["50d_return"] = compute_return(df["Close"], 50)
    df["100d_return"] = compute_return(df["Close"], 100)
    df["250d_return"] = compute_return(df["Close"], 250)

    # save into dict
    feature_data[symbol] = df

print("Feature computation finished.")

Feature computation finished.


In [17]:
combined_df = pd.concat(feature_data.values(), ignore_index=True)
combined_df.to_csv("all_stocks_with_features.csv", index=False)

print("Combined file saved.")

Combined file saved.


# Y-value preprocessing
With above features, we expect to make predictions on the stock

In [18]:
# convert columns [close, high, low] to numeric values
def clean_price_columns(df):
    df = df.copy()

    for col in ["Close", "High", "Low"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [19]:
pct_list = [2, 5, 10, 20, 50]
price_horizons = [1, 5, 20, 100, 250]
vol_horizons = [5, 20, 100, 250]

## Calculating min and max price in the next N days
These values are used to determine whether a price target was reached in the future.

In [20]:
def compute_future_window_stats(series, horizon, mode="max"):
    values = series.to_numpy()
    n = len(values)
    out = np.full(n, np.nan)

    for i in range(n):
        future_window = values[i + 1:i + 1 + horizon]

        if len(future_window) < horizon:
            continue

        if mode == "max":
            out[i] = np.nanmax(future_window)
        elif mode == "min":
            out[i] = np.nanmin(future_window)
        else:
            raise ValueError("mode must be 'max' or 'min'")

    return pd.Series(out, index=series.index)

In [21]:
def compute_future_close_hit(close, horizon, threshold, direction="up"):
    close_values = close.to_numpy()
    threshold_values = threshold.to_numpy()
    n = len(close_values)
    out = np.full(n, np.nan)

    for i in range(n):
        future_closes = close_values[i + 1:i + 1 + horizon]

        if len(future_closes) < horizon:
            continue

        if direction == "up":
            out[i] = int(np.any(future_closes >= threshold_values[i]))
        elif direction == "down":
            out[i] = int(np.any(future_closes <= threshold_values[i]))
        else:
            raise ValueError("direction must be 'up' or 'down'")

    return pd.Series(out, index=close.index)

## Volatility
Volatility measures how much a stock price move up and down over a period of time, so it tells us how risky and how large future price movements might be and we may determine trading strategies based on volativity.
- high volatility $\to$ prices move a lot
- low volatility $\to$ prices move a little

In [22]:
def compute_future_volatility(close, horizon):
    future_ret = close.pct_change()
    values = future_ret.to_numpy()
    n = len(values)
    out = np.full(n, np.nan)

    for i in range(n):
        future_window = values[i + 1:i + 1 + horizon]

        if len(future_window) < horizon:
            continue

        out[i] = np.nanstd(future_window, ddof=1)

    return pd.Series(out, index=close.index)

## High Reach and Low Reach
Target High Reach and Target Low Reach are future-based binary labels that tell you whether the price reaches a certain level within a future time window.

In [23]:
def add_high_reach_targets(df):
    df = df.copy()

    close = df["Close"]
    high = df["High"]

    for horizon in price_horizons:
        future_max_high = compute_future_window_stats(high, horizon, mode="max")

        for pct in pct_list:
            threshold = close * (1 + pct / 100)

            df[f"price_target_high_reach_{horizon}d_{pct}pct"] = np.where(
                future_max_high.notna(),
                (future_max_high >= threshold).astype(int),
                np.nan
            )

    return df
def add_low_reach_targets(df):
    df = df.copy()

    close = df["Close"]
    low = df["Low"]

    for horizon in price_horizons:
        future_min_low = compute_future_window_stats(low, horizon, mode="min")

        for pct in pct_list:
            threshold = close * (1 - pct / 100)

            df[f"price_target_low_reach_{horizon}d_{pct}pct"] = np.where(
                future_min_low.notna(),
                (future_min_low <= threshold).astype(int),
                np.nan
            )

    return df

## Close Up and Close Down
Close Up and Close Down are similar to High Reach and Low Reach, but they use future closing prices instead of future high/low prices.

In [24]:
def add_close_up_targets(df):
    df = df.copy()

    close = df["Close"]

    for horizon in price_horizons:
        for pct in pct_list:
            threshold = close * (1 + pct / 100)

            df[f"price_target_close_up_{horizon}d_{pct}pct"] = compute_future_close_hit(
                close, horizon, threshold, direction="up"
            )

    return df
def add_close_down_targets(df):
    df = df.copy()

    close = df["Close"]

    for horizon in price_horizons:
        for pct in pct_list:
            threshold = close * (1 - pct / 100)

            df[f"price_target_close_down_{horizon}d_{pct}pct"] = compute_future_close_hit(
                close, horizon, threshold, direction="down"
            )

    return df

In [25]:
def add_volatility_targets(df):
    df = df.copy()

    close = df["Close"]

    for horizon in vol_horizons:
        df[f"volatility_target_{horizon}d"] = compute_future_volatility(close, horizon)

    return df

## Combine all targets

In [26]:
def compute_targets_for_df(df):
    df = df.copy()

    df = clean_price_columns(df)

    df = add_high_reach_targets(df)
    df = add_low_reach_targets(df)
    df = add_close_up_targets(df)
    df = add_close_down_targets(df)
    df = add_volatility_targets(df)

    return df

In [27]:
combined_df = []
stock_data_with_targets = {}

for symbol, df in stock_data.items():
    try:
        df_with_targets = compute_targets_for_df(df)
        stock_data_with_targets[symbol] = df_with_targets
        print(f"Finished {symbol}")
    except Exception as e:
        print(f"Error processing {symbol}: {e}")

for symbol, df in stock_data_with_targets.items():
    
    df = df.copy()
    
    # Add stock symbol column
    df["Symbol"] = symbol
    
    # Drop unwanted columns if they exist
    drop_cols = ["Date", "Close", "High", "Low", "Open", "Volume"]
    df = df.drop(columns=[col for col in drop_cols if col in df.columns])
    
    combined_df.append(df)

# Combine everything
final_df = pd.concat(combined_df, axis=0, ignore_index=True)

# Save one combined file
final_df.to_csv("all_stocks_targets.csv", index=False)

print("Saved combined file: all_stocks_targets.csv")

Finished AMD
Finished GLD
Finished GS
Finished INTC
Finished JPM
Finished META
Finished MSFT
Finished MU
Finished NVDA
Finished RXRX
Finished TSLA
Saved combined file: all_stocks_targets.csv
